# Mejor modelo XGBoost Multicanal - paquete para inferencia

Este notebook entrena el mejor modelo XGBoost multicanal y genera los archivos necesarios para compartirlo con otra persona. No registra nada en servicios externos; todo queda guardado localmente.

Artefactos principales:

- `.joblib`: paquete completo para inferencia.
- `.json`: modelo nativo de XGBoost como respaldo.

Al final se recarga el `.joblib` y se ejecuta un `predict` de validación para confirmar que el artefacto funciona.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
)
from xgboost import XGBClassifier

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data.segmentation import CLASS_NAMES, INT_TO_CLASS

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

## Configuración

Se usan las mismas transformaciones del notebook 08/09: se excluyen identificadores y columnas de control, se codifica `channel_combo` con one-hot y se conservan las variables de disponibilidad de canales.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed" / "train_multicanal.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
ARTIFACT_DIR = MODEL_DIR / "xgboost_multicanal_artifacts"
MODEL_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_multicanal.csv"
TEST_PATH = DATA_DIR / "test_multicanal.csv"

TARGET_COLUMN = "y"
ID_COLUMNS = ["recording_id", "subject_code"]
CONTROL_COLUMNS = ["split", "epoch_idx_in_recording"]
CATEGORICAL_COLUMNS = ["channel_combo"]

MODEL_JSON_PATH = ARTIFACT_DIR / "best-xgboost-multicanal-model.json"
MODEL_JOBLIB_PATH = ARTIFACT_DIR / "best-xgboost-multicanal.joblib"
MODEL_INFO_PATH = ARTIFACT_DIR / "model_info_multicanal.json"
FEATURE_IMPORTANCE_PATH = ARTIFACT_DIR / "feature_importance_multicanal.csv"
PER_CLASS_METRICS_PATH = ARTIFACT_DIR / "per_class_metrics_multicanal.csv"
CONFUSION_MATRIX_PATH = ARTIFACT_DIR / "confusion_matrix_test_multicanal.png"

TRAIN_PATH, TEST_PATH, ARTIFACT_DIR

## Carga de datos y preparación de matrices

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

missing_columns = set(train_df.columns) ^ set(test_df.columns)
if missing_columns:
    raise ValueError(f"Train y test no tienen las mismas columnas: {missing_columns}")

excluded_columns = [TARGET_COLUMN, *ID_COLUMNS, *CONTROL_COLUMNS]
base_feature_columns = [
    column
    for column in train_df.columns
    if column not in excluded_columns
]

X_train_raw = train_df[base_feature_columns].copy()
X_test_raw = test_df[base_feature_columns].copy()

X_train = pd.get_dummies(X_train_raw, columns=CATEGORICAL_COLUMNS, dtype=np.int8)
X_test = pd.get_dummies(X_test_raw, columns=CATEGORICAL_COLUMNS, dtype=np.int8)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

for df in (X_train, X_test):
    bool_columns = df.select_dtypes(include="bool").columns
    df[bool_columns] = df[bool_columns].astype(np.int8)

feature_columns = X_train.columns.tolist()

object_columns_train = X_train.select_dtypes(include="object").columns.tolist()
object_columns_test = X_test.select_dtypes(include="object").columns.tolist()
if object_columns_train or object_columns_test:
    raise ValueError(f"Columnas object no válidas: train={object_columns_train}, test={object_columns_test}")

y_train_raw = train_df[TARGET_COLUMN]
y_test_raw = test_df[TARGET_COLUMN]

y_train_names = y_train_raw.map(INT_TO_CLASS).fillna(y_train_raw).astype(str)
y_test_names = y_test_raw.map(INT_TO_CLASS).fillna(y_test_raw).astype(str)

label_encoder = LabelEncoder()
label_encoder.fit(CLASS_NAMES)
y_train = label_encoder.transform(y_train_names)
y_test = label_encoder.transform(y_test_names)
class_labels = np.array(CLASS_NAMES)

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Número de features: {len(feature_columns)}")
print(f"Clases: {class_labels.tolist()}")
print(f"Columnas excluidas: {excluded_columns}")
display(pd.Series(feature_columns, name="features"))

## Parámetros del mejor modelo

In [ ]:
best_params = {
    "reg_lambda": 0.5,
    "reg_alpha": 0.5,
    "n_estimators": 700,
    "max_depth": 6,
    "learning_rate": 0.03,
    "gamma": 0.3,
}

fixed_params = {
    "objective": "multi:softprob",
    "num_class": len(class_labels),
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

display(pd.Series(best_params, name="best_params"))

## Utilidades

In [ ]:
def save_confusion_matrix_plot(cm: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title("Matriz de confusión - mejor XGBoost multicanal")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def prepare_features_for_inference(input_df: pd.DataFrame, model_package: dict) -> pd.DataFrame:
    """Aplica al dataframe nuevo las mismas transformaciones usadas en entrenamiento."""
    base_columns = model_package["base_feature_columns"]
    categorical_columns = model_package["categorical_columns"]
    feature_columns = model_package["feature_columns"]

    missing_base_columns = [column for column in base_columns if column not in input_df.columns]
    if missing_base_columns:
        raise ValueError(f"Faltan columnas base para inferencia: {missing_base_columns}")

    X_raw = input_df[base_columns].copy()
    X = pd.get_dummies(X_raw, columns=categorical_columns, dtype=np.int8)
    X = X.reindex(columns=feature_columns, fill_value=0)

    bool_columns = X.select_dtypes(include="bool").columns
    X[bool_columns] = X[bool_columns].astype(np.int8)

    object_columns = X.select_dtypes(include="object").columns.tolist()
    if object_columns:
        raise ValueError(f"Columnas object no válidas en inferencia: {object_columns}")

    return X

## Entrenamiento, evaluación y guardado local

In [ ]:
xgb_model = XGBClassifier(**fixed_params, **best_params)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
cohens_kappa = cohen_kappa_score(y_test, y_pred)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test,
    y_pred,
    labels=label_encoder.transform(class_labels),
    zero_division=0,
)

per_class_metrics = pd.DataFrame(
    {
        "class": class_labels,
        "precision_per_class": precision,
        "recall_per_class": recall,
        "f1_per_class": f1,
        "support": support,
    }
)

conf_matrix = confusion_matrix(y_test, y_pred, labels=label_encoder.transform(class_labels))
conf_matrix_df = pd.DataFrame(conf_matrix, index=class_labels, columns=class_labels)

feature_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": xgb_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

model_package = {
    "model": xgb_model,
    "label_encoder": label_encoder,
    "feature_columns": feature_columns,
    "base_feature_columns": base_feature_columns,
    "excluded_columns": excluded_columns,
    "id_columns": ID_COLUMNS,
    "control_columns": CONTROL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "class_labels": class_labels.tolist(),
    "best_params": best_params,
    "fixed_params": fixed_params,
    "target_column": TARGET_COLUMN,
}

xgb_model.save_model(MODEL_JSON_PATH)
joblib.dump(model_package, MODEL_JOBLIB_PATH)

save_confusion_matrix_plot(conf_matrix_df, CONFUSION_MATRIX_PATH)
per_class_metrics.to_csv(PER_CLASS_METRICS_PATH, index=False)
feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

model_info = {
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "balanced_accuracy": balanced_accuracy,
    "cohens_kappa": cohens_kappa,
    "best_params": best_params,
    "fixed_params": fixed_params,
    "n_features": len(feature_columns),
    "feature_columns": feature_columns,
    "excluded_columns": excluded_columns,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "artifact_files": {
        "xgboost_json": str(MODEL_JSON_PATH),
        "joblib_package": str(MODEL_JOBLIB_PATH),
        "confusion_matrix": str(CONFUSION_MATRIX_PATH),
        "feature_importance": str(FEATURE_IMPORTANCE_PATH),
        "per_class_metrics": str(PER_CLASS_METRICS_PATH),
    },
}
with open(MODEL_INFO_PATH, "w") as f:
    json.dump(model_info, f, indent=4)

print(f"accuracy: {accuracy:.4f}")
print(f"macro_f1: {macro_f1:.4f}")
print(f"balanced_accuracy: {balanced_accuracy:.4f}")
print(f"cohens_kappa: {cohens_kappa:.4f}")
print(f"JOBLIB guardado en: {MODEL_JOBLIB_PATH}")
display(per_class_metrics)
display(conf_matrix_df)

## Validación del artefacto guardado

In [ ]:
loaded_package = joblib.load(MODEL_JOBLIB_PATH)

validation_sample = test_df.sample(n=20, random_state=42).copy()
X_validation_sample = prepare_features_for_inference(validation_sample, loaded_package)

loaded_model = loaded_package["model"]
loaded_label_encoder = loaded_package["label_encoder"]

sample_pred_encoded = loaded_model.predict(X_validation_sample)
sample_pred_labels = loaded_label_encoder.inverse_transform(sample_pred_encoded)

validation_result = validation_sample[["recording_id", "subject_code", "channel_combo", TARGET_COLUMN]].copy()
validation_result["predicted_y"] = sample_pred_labels
validation_result["is_correct"] = validation_result[TARGET_COLUMN].astype(str) == validation_result["predicted_y"].astype(str)

print(f"Artefacto cargado desde: {MODEL_JOBLIB_PATH}")
print(f"Shape usado para predict: {X_validation_sample.shape}")
print(f"Predicciones generadas: {len(sample_pred_labels)}")
display(validation_result)

## Visualización de resultados

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de confusión - mejor XGBoost multicanal")
plt.xlabel("Predicción")
plt.ylabel("Clase real")
plt.tight_layout()

display(conf_matrix_df)

In [ ]:
plt.figure(figsize=(10, 10))
sns.barplot(data=feature_importance.head(30), x="importance", y="feature", color="#4C78A8")
plt.title("Top 30 importancia de variables - mejor XGBoost multicanal")
plt.xlabel("Importancia")
plt.ylabel("")
plt.tight_layout()

display(feature_importance)